In [1]:
from pathlib import Path
import sys

if Path('/content').exists():
    ROOT = Path('/content/woodbench')
else:
    ROOT = Path.cwd()
    if not (ROOT / 'src').exists():
        candidate = ROOT.parents[2]
        if (candidate / 'src').exists():
            ROOT = candidate

sys.path.insert(0, str(ROOT))

import torch
print(f'[ROOT] {ROOT}')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')


[ROOT] /home/sakana/Code/ManifoldsTeam/woodbench
PyTorch: 2.12.0+cu130
CUDA: True
GPU: NVIDIA GeForce RTX 3060
VRAM: 12.5 GB


# Final-B — Re-evaluate all best checkpoints

Notebook này đánh giá lại toàn bộ 30 run Final-B 17 lớp. Khác với notebook training lịch sử, mỗi run luôn tạo model mới và load `*_best.pth` trước khi inference; `*_last.pth` không bao giờ được sử dụng.

Mỗi run lưu đầy đủ `y_true`, `y_pred`, xác suất softmax, thứ tự class và đường dẫn ảnh để metric có thể tính lại mà không cần chạy model lần nữa.

In [2]:
# Colab-only dependency setup
if Path('/content').exists():
    !pip install -q -r {ROOT}/requirements.txt


In [3]:
import gc
import hashlib
import json
import platform
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from torch.utils.data import DataLoader

from src.data import FlatConiferDataset
from src.data.transforms import get_val_transforms
from src.eval.metrics import TASK_B_LABELS, TASK_B_NUM_CLASSES, compute_task_b_metrics
from src.models import create_model

DATA_DIR = ROOT / 'data' / 'task-B-group-split'
EXPERIMENT_DIR = ROOT / 'data' / 'experiments' / 'Task-B'
OUTPUT_DIR = ROOT / 'data' / 'experiments' / 'phase_1b' / 'best_checkpoint_evaluation'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_CLASSES = TASK_B_NUM_CLASSES
EXPECTED_TEST_IMAGES = 532
EXPECTED_RUNS = 30
BATCH_SIZE = 64
NUM_WORKERS = 4
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Data: {DATA_DIR}')
print(f'Checkpoints: {EXPERIMENT_DIR}')
print(f'Output: {OUTPUT_DIR}')
print(f'Device: {DEVICE}')


Data: /home/sakana/Code/ManifoldsTeam/woodbench/data/task-B-group-split
Checkpoints: /home/sakana/Code/ManifoldsTeam/woodbench/data/experiments/Task-B
Output: /home/sakana/Code/ManifoldsTeam/woodbench/data/experiments/phase_1b/best_checkpoint_evaluation
Device: cuda


In [4]:
if not DATA_DIR.exists():
    raise FileNotFoundError(
        f'{DATA_DIR} not found. Prepare the strict Final-B 17-class dataset first.'
    )
if not EXPERIMENT_DIR.exists():
    raise FileNotFoundError(f'{EXPERIMENT_DIR} not found')

test_dataset = FlatConiferDataset(
    data_root=DATA_DIR,
    split='test',
    transform=get_val_transforms(img_size=224),
    task='taskB',
)
class_names = list(test_dataset.classes)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=DEVICE.type == 'cuda',
)

assert len(class_names) == EXPECTED_CLASSES, (len(class_names), class_names)
assert len(test_dataset) == EXPECTED_TEST_IMAGES, len(test_dataset)
assert all((DATA_DIR / split / name).is_dir() for split in ['train', 'val', 'test'] for name in class_names)

print(f'Test dataset: {len(test_dataset)} images, {len(class_names)} classes')
print(json.dumps(class_names, indent=2, ensure_ascii=False))


Test dataset: 532 images, 17 classes
[
  "Araucaria araucana",
  "Callitris columellaris",
  "Calocedrus rupestris",
  "Chamaecyparis obtusa",
  "Cryptomeria japonica",
  "Cunninghamia lanceolata",
  "Fokienia hodginsii",
  "Picea glauca",
  "Pinus cernua",
  "Pinus dalatensis",
  "Pinus krempfii",
  "Pinus radiata",
  "Podocarpus pilgeri",
  "Pseudotsuga brevifolia",
  "Taxus wallichiana",
  "Thujopsis dolabrata",
  "Xanthocyparis vietnamensis"
]


In [5]:
def discover_best_checkpoints():
    records = []
    for checkpoint in sorted(EXPERIMENT_DIR.rglob('*_best.pth')):
        run_dir = checkpoint.parent
        config_path = run_dir / 'run_config.json'
        old_metrics_path = run_dir / 'test_metrics.json'
        if not config_path.exists():
            raise FileNotFoundError(f'Missing config for {checkpoint}')

        config = json.loads(config_path.read_text(encoding='utf-8'))
        old_metrics = (
            json.loads(old_metrics_path.read_text(encoding='utf-8'))
            if old_metrics_path.exists()
            else {}
        )
        record = {
            'experiment': config['experiment'],
            'seed': int(config['seed']),
            'backbone': config['backbone'],
            'num_classes': int(config['num_classes']),
            'classes': list(config['classes']),
            'checkpoint': checkpoint,
            'config_path': config_path,
            'old_metrics_path': old_metrics_path if old_metrics_path.exists() else None,
            'old_metrics': old_metrics,
        }
        records.append(record)

    keys = [(r['experiment'], r['seed'], r['backbone']) for r in records]
    assert len(records) == EXPECTED_RUNS, f'Expected {EXPECTED_RUNS} best checkpoints, found {len(records)}'
    assert len(set(keys)) == len(keys), 'Duplicate run keys found'
    assert all(r['checkpoint'].name.endswith('_best.pth') for r in records)
    assert all('_last.pth' not in r['checkpoint'].name for r in records)
    assert all(r['num_classes'] == EXPECTED_CLASSES for r in records)
    assert all(r['classes'] == class_names for r in records), 'Checkpoint class order differs from test dataset'
    return records

run_records = discover_best_checkpoints()
print(f'Discovered {len(run_records)} best checkpoints')
pd.DataFrame(
    [{k: v for k, v in row.items() if k not in {'classes', 'old_metrics'}} for row in run_records]
)[['experiment', 'seed', 'backbone', 'num_classes', 'checkpoint']]


Discovered 30 best checkpoints


,experiment,seed,backbone,num_classes,checkpoint
0,main,123,convnext_tiny,17,/home/sakana/Code/ManifoldsTeam/woodbench/data...
1,main,123,efficientnet_b3,17,/home/sakana/Code/ManifoldsTeam/woodbench/data...
2,main,123,mobilenetv3_large,17,/home/sakana/Code/ManifoldsTeam/woodbench/data...
3,main,123,resnet50,17,/home/sakana/Code/ManifoldsTeam/woodbench/data...
4,main,123,swin_tiny,17,/home/sakana/Code/ManifoldsTeam/woodbench/data...
5,main,42,convnext_tiny,17,/home/sakana/Code/ManifoldsTeam/woodbench/data...
6,main,42,efficientnet_b3,17,/home/sakana/Code/ManifoldsTeam/woodbench/data...
7,main,42,mobilenetv3_large,17,/home/sakana/Code/ManifoldsTeam/woodbench/data...
8,main,42,resnet50,17,/home/sakana/Code/ManifoldsTeam/woodbench/data...
9,main,42,swin_tiny,17,/home/sakana/Code/ManifoldsTeam/woodbench/data...


In [6]:
def load_checkpoint(path):
    try:
        payload = torch.load(path, map_location='cpu', weights_only=False)
    except TypeError:
        payload = torch.load(path, map_location='cpu')
    state_dict = payload.get('model_state_dict', payload)
    return payload, state_dict


def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


@torch.inference_mode()
def evaluate_best_checkpoint(record):
    checkpoint = record['checkpoint']
    payload, state_dict = load_checkpoint(checkpoint)
    model = create_model(
        record['backbone'],
        num_classes=EXPECTED_CLASSES,
        pretrained=False,
    )
    model.load_state_dict(state_dict, strict=True)
    model = model.to(DEVICE).eval()

    y_true_batches = []
    y_pred_batches = []
    probability_batches = []
    for images, targets in test_loader:
        logits = model(images.to(DEVICE, non_blocking=True))
        probabilities = torch.softmax(logits, dim=1)
        y_true_batches.append(targets.numpy())
        y_pred_batches.append(probabilities.argmax(dim=1).cpu().numpy())
        probability_batches.append(probabilities.cpu().numpy().astype(np.float32))

    y_true = np.concatenate(y_true_batches).astype(np.int64)
    y_pred = np.concatenate(y_pred_batches).astype(np.int64)
    probabilities = np.concatenate(probability_batches).astype(np.float32)
    labels = list(TASK_B_LABELS)
    task_b_metrics = compute_task_b_metrics(
        y_true, y_pred, class_names=class_names
    )

    assert y_true.shape == (EXPECTED_TEST_IMAGES,)
    assert y_pred.shape == y_true.shape
    assert probabilities.shape == (EXPECTED_TEST_IMAGES, EXPECTED_CLASSES)
    assert sorted(np.unique(y_true).tolist()) == labels
    assert np.allclose(probabilities.sum(axis=1), 1.0, atol=1e-5)

    run_output = OUTPUT_DIR / record['experiment'] / f"seed_{record['seed']}" / record['backbone']
    run_output.mkdir(parents=True, exist_ok=True)
    np.save(run_output / 'y_true.npy', y_true)
    np.save(run_output / 'y_pred.npy', y_pred)
    np.save(run_output / 'probabilities.npy', probabilities)
    np.save(run_output / 'confusion_matrix.npy', task_b_metrics['confusion_matrix'])
    (run_output / 'class_names.json').write_text(
        json.dumps(class_names, indent=2, ensure_ascii=False), encoding='utf-8'
    )
    sample_paths = [str(path.relative_to(ROOT)) for path, _ in test_dataset.samples]
    (run_output / 'sample_paths.json').write_text(
        json.dumps(sample_paths, indent=2, ensure_ascii=False), encoding='utf-8'
    )

    old = record['old_metrics']
    metrics = {
        'experiment': record['experiment'],
        'seed': record['seed'],
        'backbone': record['backbone'],
        'num_classes': EXPECTED_CLASSES,
        'test_images': int(len(y_true)),
        'checkpoint': str(checkpoint.relative_to(ROOT)),
        'checkpoint_kind': 'best',
        'checkpoint_sha256': sha256_file(checkpoint),
        'checkpoint_epoch': int(payload.get('epoch', -1)) + 1,
        'best_epoch': int(payload.get('best_epoch', payload.get('epoch', -1))) + 1,
        'best_val_acc': float(payload.get('best_val_acc', np.nan)),
        'accuracy': task_b_metrics['accuracy'],
        'precision_macro': task_b_metrics['precision_macro'],
        'recall_macro': task_b_metrics['recall_macro'],
        'f1_macro': task_b_metrics['f1_macro'],
        'old_notebook_accuracy': float(old.get('test_accuracy', np.nan)),
        'old_notebook_f1_macro': float(old.get('test_f1_macro', np.nan)),
    }
    metrics['delta_accuracy_pp'] = 100 * (metrics['accuracy'] - metrics['old_notebook_accuracy'])
    metrics['delta_f1_pp'] = 100 * (metrics['f1_macro'] - metrics['old_notebook_f1_macro'])
    (run_output / 'metrics.json').write_text(
        json.dumps(metrics, indent=2, ensure_ascii=False), encoding='utf-8'
    )

    del model, payload, state_dict
    gc.collect()
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
    return metrics


In [7]:
results = []
for index, record in enumerate(run_records, start=1):
    metrics = evaluate_best_checkpoint(record)
    results.append(metrics)
    print(
        f"[{index:02d}/{len(run_records)}] {metrics['experiment']:<26s} "
        f"{metrics['backbone']:<20s} seed={metrics['seed']:<3d} "
        f"Acc={metrics['accuracy']*100:7.3f}% F1={metrics['f1_macro']*100:7.3f}% "
        f"dAcc={metrics['delta_accuracy_pp']:+7.3f} pp"
    )

results_df = pd.DataFrame(results).sort_values(
    ['experiment', 'backbone', 'seed']
).reset_index(drop=True)
results_df


[01/30] main                       convnext_tiny        seed=123 Acc= 99.436% F1= 99.483% dAcc= -0.188 pp


[02/30] main                       efficientnet_b3      seed=123 Acc= 97.368% F1= 97.448% dAcc= +0.000 pp


[03/30] main                       mobilenetv3_large    seed=123 Acc= 98.684% F1= 98.728% dAcc= +0.188 pp


[04/30] main                       resnet50             seed=123 Acc= 92.293% F1= 92.490% dAcc= +0.000 pp


[05/30] main                       swin_tiny            seed=123 Acc= 99.812% F1= 99.827% dAcc= -0.188 pp


[06/30] main                       convnext_tiny        seed=42  Acc= 99.624% F1= 99.608% dAcc= +0.000 pp


... (21 outputs left out)

[28/30] taskB_label_smoothing_0.2  mobilenetv3_large    seed=42  Acc= 98.684% F1= 98.726% dAcc= +0.188 pp


[29/30] taskB_label_smoothing_0.2  resnet50             seed=42  Acc= 93.985% F1= 94.090% dAcc= -0.188 pp


[30/30] taskB_label_smoothing_0.2  swin_tiny            seed=42  Acc= 99.624% F1= 99.656% dAcc= +0.188 pp


,experiment,seed,backbone,num_classes,test_images,checkpoint,checkpoint_kind,checkpoint_sha256,checkpoint_epoch,best_epoch,best_val_acc,accuracy,precision_macro,recall_macro,f1_macro,old_notebook_accuracy,old_notebook_f1_macro,delta_accuracy_pp,delta_f1_pp
0,main,42,convnext_tiny,17,532,data/experiments/Task-B/main/seed_42/convnext_...,best,962ba72f75059163bc630bc7efe407323da0031753fa5b...,20,20,99.889258,0.996241,0.996379,0.995915,0.996081,0.996241,0.996537,0.000000,-0.045555
1,main,123,convnext_tiny,17,532,data/experiments/Task-B/main/seed_123/convnext...,best,77b51b616a0047444f69d19a35ee331e92869fac903387...,16,16,99.889258,0.994361,0.994810,0.995098,0.994830,0.996241,0.996560,-0.187970,-0.172973
2,main,456,convnext_tiny,17,532,data/experiments/Task-B/main/seed_456/convnext...,best,3524c7b706434b7b8fb4da197ae9bccdc67c15e3b0fb9b...,14,14,99.889258,0.994361,0.994958,0.995098,0.994809,0.919173,0.917272,7.518797,7.753637
3,main,42,efficientnet_b3,17,532,data/experiments/Task-B/main/seed_42/efficient...,best,820c6410d4a316e842171feebca52d450a0f4e94acb221...,32,32,95.791805,0.969925,0.971164,0.970001,0.969505,0.973684,0.974155,-0.375940,-0.465049
4,main,123,efficientnet_b3,17,532,data/experiments/Task-B/main/seed_123/efficien...,best,b00c29ce65bab5de75f5a487446afddcb54186ec7e2853...,38,38,96.566999,0.973684,0.977088,0.973950,0.974476,0.973684,0.974168,0.000000,0.030818
5,main,456,efficientnet_b3,17,532,data/experiments/Task-B/main/seed_456/efficien...,best,029951432391da4dfe4c84fe13487d370b67913f193c40...,27,27,96.234773,0.969925,0.972234,0.970382,0.970152,0.973684,0.974168,-0.375940,-0.401590
6,main,42,mobilenetv3_large,17,532,data/experiments/Task-B/main/seed_42/mobilenet...,best,a54ac31c4dfc62129671c6df100c387d63c79e2ed61f6e...,43,43,95.348837,0.984962,0.986688,0.985263,0.985533,0.986842,0.987261,-0.187970,-0.172755
7,main,123,mobilenetv3_large,17,532,data/experiments/Task-B/main/seed_123/mobilene...,best,4a4a179ea0d656649b86a44fe755646fced565e8414644...,47,47,95.570321,0.986842,0.988253,0.987101,0.987280,0.984962,0.985538,0.187970,0.174233
8,main,456,mobilenetv3_large,17,532,data/experiments/Task-B/main/seed_456/mobilene...,best,a3f1a6340bebf9ffb98d83d7bc49d0f99f7d75ab275f68...,38,38,95.570321,0.979323,0.981368,0.978900,0.979385,0.986842,0.987487,-0.751880,-0.810137
9,main,42,resnet50,17,532,data/experiments/Task-B/main/seed_42/resnet50/...,best,3e2627481c0bdcdc8aa555181326a87e483e4ef8f47eee...,50,50,90.586932,0.947368,0.953093,0.948177,0.948664,0.945489,0.946905,0.187970,0.175930


In [8]:
assert len(results_df) == EXPECTED_RUNS
assert results_df['checkpoint_kind'].eq('best').all()
assert results_df['checkpoint'].str.endswith('_best.pth').all()
assert results_df['num_classes'].eq(EXPECTED_CLASSES).all()
assert results_df['test_images'].eq(EXPECTED_TEST_IMAGES).all()

main_df = results_df[results_df['experiment'].eq('main')]
assert len(main_df) == 15
main_summary_df = (
    main_df.groupby('backbone')
    .agg(
        runs=('seed', 'count'),
        accuracy_mean=('accuracy', 'mean'),
        accuracy_std=('accuracy', lambda values: values.std(ddof=0)),
        f1_mean=('f1_macro', 'mean'),
        f1_std=('f1_macro', lambda values: values.std(ddof=0)),
    )
    .reset_index()
)

comparison_df = results_df[[
    'experiment', 'seed', 'backbone',
    'old_notebook_accuracy', 'accuracy', 'delta_accuracy_pp',
    'old_notebook_f1_macro', 'f1_macro', 'delta_f1_pp',
]].copy()
comparison_df = comparison_df.reindex(
    comparison_df['delta_accuracy_pp'].abs().sort_values(ascending=False).index
)

results_df.to_csv(OUTPUT_DIR / 'best_checkpoint_metrics.csv', index=False)
main_summary_df.to_csv(OUTPUT_DIR / 'best_checkpoint_main_summary.csv', index=False)
comparison_df.to_csv(OUTPUT_DIR / 'notebook_vs_best_comparison.csv', index=False)
(OUTPUT_DIR / 'best_checkpoint_metrics.json').write_text(
    json.dumps(results_df.to_dict(orient='records'), indent=2, ensure_ascii=False),
    encoding='utf-8',
)

print('Main 3-seed summary (best checkpoints only)')
display(main_summary_df)
print('Largest notebook-vs-best differences')
display(comparison_df.head(10))


Main 3-seed summary (best checkpoints only)


,backbone,runs,accuracy_mean,accuracy_std,f1_mean,f1_std
0,convnext_tiny,3,0.994987,0.000886,0.995240,0.000595
1,efficientnet_b3,3,0.971178,0.001772,0.971378,0.002207
2,mobilenetv3_large,3,0.983709,0.003195,0.984066,0.003386
3,resnet50,3,0.936717,0.010219,0.938440,0.009978
4,swin_tiny,3,0.994361,0.003070,0.994815,0.002825


Largest notebook-vs-best differences


,experiment,seed,backbone,old_notebook_accuracy,accuracy,delta_accuracy_pp,old_notebook_f1_macro,f1_macro,delta_f1_pp
2,main,456,convnext_tiny,0.919173,0.994361,7.518797,0.917272,0.994809,7.753637
12,main,42,swin_tiny,0.971805,0.990602,1.879699,0.973919,0.991348,1.742831
25,taskB_label_smoothing_0.2,42,convnext_tiny,0.990602,0.979323,-1.127820,0.991027,0.980485,-1.054215
21,taskB_label_smoothing_0.0,42,efficientnet_b3,0.971805,0.964286,-0.751880,0.971795,0.964461,-0.733342
8,main,456,mobilenetv3_large,0.986842,0.979323,-0.751880,0.987487,0.979385,-0.810137
3,main,42,efficientnet_b3,0.973684,0.969925,-0.375940,0.974155,0.969505,-0.465049
26,taskB_label_smoothing_0.2,42,efficientnet_b3,0.960526,0.956767,-0.375940,0.960587,0.956980,-0.360710
5,main,456,efficientnet_b3,0.973684,0.969925,-0.375940,0.974168,0.970152,-0.401590
11,main,456,resnet50,0.936090,0.939850,0.375940,0.937494,0.941751,0.425723
16,taskB_90rotation,42,efficientnet_b3,0.988722,0.986842,-0.187970,0.989289,0.987212,-0.207709


In [9]:
metadata = {
    'evaluated_at_utc': datetime.now(timezone.utc).isoformat(),
    'checkpoint_policy': 'Only *_best.pth; *_last.pth forbidden',
    'dataset': str(DATA_DIR.relative_to(ROOT)),
    'experiment_root': str(EXPERIMENT_DIR.relative_to(ROOT)),
    'output_root': str(OUTPUT_DIR.relative_to(ROOT)),
    'num_runs': len(results_df),
    'num_classes': len(class_names),
    'test_images': len(test_dataset),
    'class_names': class_names,
    'batch_size': BATCH_SIZE,
    'device_type': DEVICE.type,
    'device_name': torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else platform.processor() or platform.machine(),
    'python_version': platform.python_version(),
    'torch_version': torch.__version__,
    'metric_policy': {
        'average': 'macro',
        'labels': list(range(EXPECTED_CLASSES)),
        'zero_division': 0,
    },
}
(OUTPUT_DIR / 'evaluation_metadata.json').write_text(
    json.dumps(metadata, indent=2, ensure_ascii=False), encoding='utf-8'
)

expected_bundle_files = {
    'y_true.npy', 'y_pred.npy', 'probabilities.npy', 'confusion_matrix.npy',
    'class_names.json', 'sample_paths.json', 'metrics.json',
}
run_metric_files = list(OUTPUT_DIR.glob('*/seed_*/*/metrics.json'))
assert len(run_metric_files) == EXPECTED_RUNS
for metric_file in run_metric_files:
    assert expected_bundle_files.issubset({path.name for path in metric_file.parent.iterdir()})

print(f'Validated {len(run_metric_files)} complete prediction bundles')
print(f'Saved all outputs to: {OUTPUT_DIR}')


Validated 30 complete prediction bundles
Saved all outputs to: /home/sakana/Code/ManifoldsTeam/woodbench/data/experiments/phase_1b/best_checkpoint_evaluation
